In [ ]:
# ---------------------------------------------------------------------------
# Root segmentation pipeline — intensity-threshold classification with
# gray-value sand masking and slice-to-slice overlap tracking
# ---------------------------------------------------------------------------
# This script segments the root body from a micro-CT volume using a
# three-stage approach: (1) NLM denoising of the normalized intensity stack,
# (2) removal of sand voxels identified by a fixed gray-value intensity
# range followed by morphological cleaning, and (3) voxel classification
# into root and background classes using intensity thresholds derived from
# histogram analysis. The root body is then isolated from other root-class
# connected components via a slice-to-slice overlap propagation algorithm
# seeded at a manually verified anchor slice. Post-processing applies
# hole-filling, small-object removal, and 3D binary closing to produce a
# clean binary root mask ready for downstream metric evaluation.
# ---------------------------------------------------------------------------

import os
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from skimage.restoration import denoise_nl_means
from skimage.morphology import (
    binary_closing, binary_dilation,
    remove_small_objects, remove_small_holes,
    disk, ball
)
from skimage.measure import label, regionprops
from scipy.ndimage import binary_opening

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# Path to the folder of raw 2D TIFF slices for one scan session.
input_folder = "/path/to/raw/slices"

# Number of slices to load from the start of the sorted file list.
# Adjust to select the sub-volume of interest.
slice_end = 500

# NLM denoising parameter h (fixed, not scaled by sigma).
# Larger values smooth more aggressively; determined empirically.
nlm_h = 6

# Gray-value range (normalized [0, 1]) that identifies sand voxels.
# Voxels in this range are masked out before root classification.
sand_min, sand_max = 0.080, 0.106

# Morphological parameters for sand mask cleaning.
sand_closing_radius  = 4    # disk radius for binary closing (gap filling)
sand_min_object_size = 500  # minimum retained sand region area (pixels)
sand_dilation_radius = 4    # disk radius for dilation after cleaning

# Intensity thresholds for voxel classification (normalized [0, 1])
# after sand removal. Determined from histogram analysis.
root_min, root_max = 0.0555, 0.067   # gray-value range for root tissue
bg_min,   bg_max   = 0.049,  0.063   # gray-value range for background/air

# Morphological opening before connected-component labelling.
# Breaking narrow bridges reduces spurious region merging.
opening_radius = 6   # structuring element side length (pixels)

# Anchor slice: the slice index and region label known to contain the root,
# used as the seed for the overlap-tracking propagation algorithm.
anchor_z     = 1
anchor_label = 136   # replace with the actual label in labeled_slices[anchor_z]

# Post-processing parameters applied to the tracked root mask.
hole_fill_threshold  = 200   # maximum hole area (pixels) to fill per slice
min_object_size      = 150   # minimum retained object size (voxels)
closing_ball_radius  = 6     # 3D ball radius for final binary closing

# Output file paths.
output_normalized   = "normalized_stack.tif"
output_denoised     = "denoised_stack.tif"
output_no_sand      = "denoised_without_sand.tif"
output_classified   = "classified_stack.tif"
output_labeled      = "labeled_slices.tif"
output_root_raw     = "filtered_root_mask_8uint.tif"
output_root_cleaned = "filtered_root_cleaned.tif"
output_dir_sand       = "sand_mask"
output_dir_classified = "classified_slices"

# ---------------------------------------------------------------------------
# 1. Load raw slices and build 3D stack
# ---------------------------------------------------------------------------
file_list = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.tif')])
file_list = file_list[:slice_end]

slices = []
for fname in file_list:
    slices.append(tiff.imread(os.path.join(input_folder, fname)))

stack = np.stack(slices, axis=0)   # shape: (Z, Y, X)
print(f"Loaded stack shape (Z, Y, X): {stack.shape}")

# ---------------------------------------------------------------------------
# 2. Min-max normalization to [0, 1]
# ---------------------------------------------------------------------------
stack_norm = (stack - np.min(stack)) / (np.max(stack) - np.min(stack))

# Preview first, middle, and last slice to confirm correct slice range.
for i in [0, len(stack_norm) // 2, -1]:
    plt.figure(figsize=(5, 5))
    plt.imshow(stack_norm[i], cmap="gray")
    plt.axis("off")
    plt.show()

tiff.imwrite(output_normalized, stack_norm.astype(np.float32), imagej=True)
print(f"Normalised stack saved to: {os.path.abspath(output_normalized)}")

# ---------------------------------------------------------------------------
# 3. NLM denoising — full stack
# ---------------------------------------------------------------------------
# Non-Local Means filtering exploits self-similarity across spatially
# separated patches to suppress noise while preserving structural edges.
# patch_distance=3 is used here (vs. 6 in the Sav_S5 script) because this
# session's voxel size is smaller, requiring a tighter search window.
denoised_stack = np.empty_like(stack_norm)

for i in range(stack_norm.shape[0]):
    print(f"Denoising slice {i + 1}/{stack_norm.shape[0]}")
    denoised_stack[i] = denoise_nl_means(
        stack_norm[i],
        h=nlm_h,
        fast_mode=True,
        patch_size=5,
        patch_distance=3
    )

print("Denoising complete.")
tiff.imwrite(output_denoised, denoised_stack.astype(np.float32), imagej=True)
print(f"Denoised stack saved to: {os.path.abspath(output_denoised)}")

# ---------------------------------------------------------------------------
# 4. Sand segmentation by fixed intensity range
# ---------------------------------------------------------------------------
# Rather than using Otsu thresholding (which varies per slice), this session
# uses a fixed gray-value window derived from the histogram of the denoised
# volume. This is more appropriate when the sand peak is well-separated and
# stable across slices.
os.makedirs(output_dir_sand, exist_ok=True)
sand_mask_stack = np.zeros_like(denoised_stack, dtype=np.uint8)

for z in range(denoised_stack.shape[0]):
    slice_ = denoised_stack[z]
    sand_mask_stack[z] = ((slice_ >= sand_min) & (slice_ <= sand_max)).astype(np.uint8)

# Visualize sand mask on a sample of slices.
cmap_sand = mcolors.ListedColormap(["black", "red"])
norm_sand  = mcolors.BoundaryNorm([0, 0.5, 1], cmap_sand.N)

for z in range(12, 19):
    plt.figure()
    plt.imshow(sand_mask_stack[z], cmap=cmap_sand, norm=norm_sand)
    plt.title(f"Sand mask — slice {z}  (red = sand)")
    plt.axis("off")
    plt.show()

# ---------------------------------------------------------------------------
# 5. Sand mask cleaning — closing, small-object removal, dilation
# ---------------------------------------------------------------------------
# Binary closing fills gaps within sand grains caused by pore voxels whose
# intensity falls outside the sand range. Small isolated specks are removed
# as likely misclassified background. Dilation expands the mask to cover
# the grain boundary layer where mixed-material voxels are common.
cleaned_sand_stack = np.zeros_like(sand_mask_stack, dtype=bool)

for z in range(sand_mask_stack.shape[0]):
    mask    = sand_mask_stack[z].astype(bool)
    closed  = binary_closing(mask, footprint=disk(sand_closing_radius))
    cleaned = remove_small_objects(closed, min_size=sand_min_object_size)
    dilated = binary_dilation(cleaned, footprint=disk(sand_dilation_radius))
    cleaned_sand_stack[z] = dilated

# Overlay cleaned sand mask on the denoised slices for visual verification.
for z in range(0, 5):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(denoised_stack[z], cmap="gray", interpolation="none")
    ax.imshow(cleaned_sand_stack[z], cmap="Reds", alpha=0.4, interpolation="none")
    ax.set_title(f"Cleaned sand mask overlay — slice {z}")
    ax.axis("off")
    plt.show()

# ---------------------------------------------------------------------------
# 6. Remove sand from the denoised stack
# ---------------------------------------------------------------------------
# Sand voxels are set to 0.0 so that they fall below all classification
# thresholds and are assigned to the unlabelled class (0) in the next step.
no_sand_stack = np.copy(denoised_stack)
no_sand_stack[cleaned_sand_stack] = 0.0

tiff.imwrite(output_no_sand, no_sand_stack.astype(np.float32), imagej=True)
print(f"Sand-removed stack saved to: {os.path.abspath(output_no_sand)}")

# ---------------------------------------------------------------------------
# 7. Intensity-threshold classification — full stack
# ---------------------------------------------------------------------------
# Each sand-removed voxel is classified into root (1) or background/air (2)
# using fixed intensity windows from the histogram analysis. The background
# mask is applied only where root has not already been assigned to prevent
# double labelling in the overlapping intensity range. Voxels set to 0 by
# sand removal remain class 0 (unlabelled).
os.makedirs(output_dir_classified, exist_ok=True)
classified_stack = np.zeros_like(no_sand_stack, dtype=np.uint8)

for z in range(no_sand_stack.shape[0]):
    slice_ = no_sand_stack[z]
    root_mask       = (slice_ >= root_min) & (slice_ <= root_max)
    background_mask = (slice_ >= bg_min)   & (slice_ <= bg_max)

    classified = np.zeros_like(slice_, dtype=np.uint8)
    classified[root_mask] = 1
    classified[(background_mask) & (classified == 0)] = 2

    classified_stack[z] = classified
    tiff.imwrite(os.path.join(output_dir_classified, f"class_{z:04d}.tif"), classified)

# Visualize classification for an example slice.
cmap_class = mcolors.ListedColormap(["black", "green", "blue"])
norm_class  = mcolors.BoundaryNorm([0, 1, 2, 3], cmap_class.N)

for idx in range(54, 55):
    plt.figure()
    plt.imshow(classified_stack[idx], cmap=cmap_class, norm=norm_class)
    plt.title(f"Classification — slice {idx}  (green=root, blue=background)")
    plt.axis("off")
    plt.show()

tiff.imwrite(output_classified, classified_stack.astype(np.uint16), imagej=True)
print(f"Classified stack saved to: {os.path.abspath(output_classified)}")

# ---------------------------------------------------------------------------
# 8. Morphological opening and 2D connected-component labelling
# ---------------------------------------------------------------------------
# Binary opening (erosion then dilation) breaks narrow bridges between
# touching regions before labelling, reducing spurious merging of the root
# with nearby soil aggregates. Labels are assigned independently per slice.
root_binary  = (classified_stack == 1)
opened_mask  = np.zeros_like(root_binary, dtype=bool)

for z in range(root_binary.shape[0]):
    opened_mask[z] = binary_opening(root_binary[z], structure=np.ones((opening_radius, opening_radius)))

labeled_slices = np.zeros_like(opened_mask, dtype=np.int32)
for z in range(opened_mask.shape[0]):
    labeled_slices[z] = label(opened_mask[z], connectivity=2)

# Visualize labeled regions for the anchor slice.
plt.figure(figsize=(8, 8))
plt.imshow(labeled_slices[anchor_z], cmap="nipy_spectral")
plt.colorbar(label="Region label")
plt.title(f"2D connected components — slice {anchor_z}")
plt.show()

# Print the 10 largest regions to help identify the anchor label.
props_sorted = sorted(regionprops(labeled_slices[anchor_z]), key=lambda r: -r.area)
print(f"\nTop 10 regions in slice {anchor_z}:")
for i, region in enumerate(props_sorted[:10], start=1):
    print(f"  {i:2d}. Label = {region.label:4d}, Area = {region.area:7d} voxels")

# Visualize each of the top 10 regions individually.
for i, region in enumerate(props_sorted[:10], start=1):
    mask = np.zeros_like(labeled_slices[anchor_z], dtype=bool)
    mask[tuple(zip(*region.coords))] = True
    plt.figure()
    plt.imshow(mask, cmap="gray")
    plt.title(f"Region {i}: label={region.label}, area={region.area}")
    plt.axis("off")
    plt.show()

tiff.imwrite(output_labeled, labeled_slices.astype(np.uint16))
print(f"Labeled slices saved to: {os.path.abspath(output_labeled)}")

# ---------------------------------------------------------------------------
# 9. Root isolation — slice-to-slice overlap tracking
# ---------------------------------------------------------------------------
# Starting from the anchor slice and region, the root body is traced upward
# and downward through the stack. At each step the region in the adjacent
# slice with the greatest pixel overlap with the current region is accepted
# as the root continuation. Tracking stops when no region of at least 50
# voxels overlaps, or when the stack boundary is reached.

def find_best_overlap_region(labeled_slice, prev_coords):
    """Return the coordinates of the region in labeled_slice that overlaps
    most with the pixel coordinates prev_coords from the previous slice.

    Parameters
    ----------
    labeled_slice : ndarray of int, shape (Y, X)
        Per-slice label image from skimage.measure.label.
    prev_coords : ndarray of int, shape (N, 2)
        (row, col) coordinates of the accepted region in the previous slice.

    Returns
    -------
    best_coords : ndarray of int, shape (M, 2) or None
        Coordinates of the best-matching region, or None if no overlap found.
    """
    prev_mask = np.zeros_like(labeled_slice, dtype=bool)
    prev_mask[tuple(prev_coords.T)] = True

    best_overlap = 0
    best_coords  = None

    for region in regionprops(labeled_slice):
        region_mask = np.zeros_like(labeled_slice, dtype=bool)
        region_mask[region.coords[:, 0], region.coords[:, 1]] = True
        overlap = np.logical_and(prev_mask, region_mask).sum()
        if overlap > best_overlap:
            best_overlap = overlap
            best_coords  = region.coords

    return best_coords


filtered_root_mask = np.zeros_like(labeled_slices, dtype=bool)
seed_coords = np.argwhere(labeled_slices[anchor_z] == anchor_label)
filtered_root_mask[anchor_z][tuple(seed_coords.T)] = True

for direction in [-1, 1]:   # -1 = toward slice 0, +1 = toward last slice
    z              = anchor_z
    current_coords = np.argwhere(filtered_root_mask[z])

    while 0 <= z + direction < labeled_slices.shape[0]:
        z += direction
        if labeled_slices[z].max() == 0:
            break   # empty slice — stop tracking in this direction

        best_coords = find_best_overlap_region(labeled_slices[z], current_coords)

        if best_coords is None or len(best_coords) < 50:
            break   # no convincing continuation found

        filtered_root_mask[z][tuple(best_coords.T)] = True
        current_coords = best_coords

# Save raw tracked mask.
tiff.imwrite(output_root_raw, (filtered_root_mask * 255).astype(np.uint8))
print(f"Raw tracked root mask saved to: {os.path.abspath(output_root_raw)}")

# ---------------------------------------------------------------------------
# 10. Post-processing — hole filling, small-object removal, 3D closing
# ---------------------------------------------------------------------------
# Small holes within the root cross-section (caused by air-filled tissue
# channels or thresholding gaps) are filled per slice. Isolated specks
# smaller than min_object_size are removed as segmentation noise. Finally,
# 3D binary closing with a ball structuring element connects small gaps
# between adjacent slices that were missed by the overlap tracker.
cleaned_mask = np.copy(filtered_root_mask)

for z in range(filtered_root_mask.shape[0]):
    cleaned_mask[z] = remove_small_holes(cleaned_mask[z], area_threshold=hole_fill_threshold)

cleaned_mask = remove_small_objects(cleaned_mask, min_size=min_object_size)
closed_mask  = binary_closing(cleaned_mask, footprint=ball(closing_ball_radius))

tiff.imwrite(output_root_cleaned, closed_mask.astype(np.uint8) * 255)
print(f"Cleaned root mask saved to: {os.path.abspath(output_root_cleaned)}")